# 03.5 - Correlation & Regression Basics

**Phase:** 03 - Statistics & Probability
**Status:** VERIFIED
---

## What Are We Solving?
Correlation measures linear association. Regression models the relationship between variables.

## Mental Model
Correlation = how much two variables move together. Regression = predicting one from the other.

## Core Concepts
- Pearson correlation (linear)
- Spearman correlation (monotonic)
- simple linear regression
- coefficient interpretation
- R²
- assumptions: linearity, independence, homoscedasticity, normality

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)

# Generate correlated data
n = 200
x = np.random.normal(0, 1, n)
y_linear = 2 * x + np.random.normal(0, 0.5, n)  # Strong linear
y_nonlinear = x**2 + np.random.normal(0, 0.5, n)  # Non-linear

print("=== CORRELATION ===")
print(f"Linear relationship:")
print(f"  Pearson r: {stats.pearsonr(x, y_linear)[0]:.3f}")
print(f"  Spearman rho: {stats.spearmanr(x, y_linear)[0]:.3f}")

print(f"\nNon-linear (quadratic) relationship:")
print(f"  Pearson r: {stats.pearsonr(x, y_nonlinear)[0]:.3f}")
print(f"  Spearman rho: {stats.spearmanr(x, y_nonlinear)[0]:.3f}")

=== CORRELATION ===
Linear relationship:
  Pearson r: 0.968
  Spearman rho: 0.966

Non-linear (quadratic) relationship:
  Pearson r: -0.021
  Spearman rho: -0.106


In [2]:
# Visualize correlations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(x, y_linear, alpha=0.6)
axes[0].set_title(f'Linear: Pearson={stats.pearsonr(x, y_linear)[0]:.2f}')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Add regression line
x_sorted = np.sort(x)
y_pred = 2 * x_sorted
axes[0].plot(x_sorted, y_pred, 'r-', lw=2, label='True line')
axes[0].legend()

axes[1].scatter(x, y_nonlinear, alpha=0.6)
axes[1].set_title(f'Quadratic: Pearson={stats.pearsonr(x, y_nonlinear)[0]:.2f}, Spearman={stats.spearmanr(x, y_nonlinear)[0]:.2f}')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')

# Spurious correlation
temp = np.random.normal(25, 5, n)
ice_cream = 2 * temp + np.random.normal(0, 3, n)
drowning = 0.5 * temp + np.random.normal(0, 2, n)
axes[2].scatter(ice_cream, drowning, alpha=0.6)
axes[2].set_title(f'Spurious (both driven by temp): Pearson={stats.pearsonr(ice_cream, drowning)[0]:.2f}')
axes[2].set_xlabel('Ice cream sales')
axes[2].set_ylabel('Drowning')

plt.tight_layout()
plt.savefig('correlation_examples.png', dpi=150, bbox_inches='tight')
print("Saved: correlation_examples.png")

Saved: correlation_examples.png


## Simple Linear Regression
y = β₀ + β₁x + ε
- β₀: intercept (y when x=0)
- β₁: slope (change in y per unit change in x)
- R²: proportion of variance explained

In [3]:
# Fit linear regression
X = x.reshape(-1, 1)
model = LinearRegression()
model.fit(X, y_linear)

y_pred = model.predict(X)
r2 = r2_score(y_linear, y_pred)
mse = mean_squared_error(y_linear, y_pred)

print(f"=== LINEAR REGRESSION ===")
print(f"True: y = 2x + noise")
print(f"Fitted: y = {model.intercept_:.3f} + {model.coef_[0]:.3f}x")
print(f"R² = {r2:.3f}")
print(f"MSE = {mse:.3f}")
print(f"RMSE = {np.sqrt(mse):.3f}")

# Coefficient interpretation
print(f"\nInterpretation:")
print(f"  Intercept ({model.intercept_:.3f}): expected y when x=0")
print(f"  Slope ({model.coef_[0]:.3f}): for each 1 unit increase in x, y increases by {model.coef_[0]:.3f}")

=== LINEAR REGRESSION ===
True: y = 2x + noise
Fitted: y = 0.045 + 2.050x
R² = 0.938
MSE = 0.240
RMSE = 0.490

Interpretation:
  Intercept (0.045): expected y when x=0
  Slope (2.050): for each 1 unit increase in x, y increases by 2.050


In [4]:
# Visualize regression with confidence intervals
from scipy import stats

# Manual calculation for CI
n = len(x)
x_mean = x.mean()
Sxx = np.sum((x - x_mean)**2)
residuals = y_linear - y_pred
sigma2 = np.sum(residuals**2) / (n - 2)
se_slope = np.sqrt(sigma2 / Sxx)
se_intercept = np.sqrt(sigma2 * (1/n + x_mean**2 / Sxx))

# 95% CI for slope
t_crit = stats.t.ppf(0.975, n-2)
slope_ci = (model.coef_[0] - t_crit * se_slope, model.coef_[0] + t_crit * se_slope)
intercept_ci = (model.intercept_ - t_crit * se_intercept, model.intercept_ + t_crit * se_intercept)

print(f"\n95% CI for slope: [{slope_ci[0]:.3f}, {slope_ci[1]:.3f}]")
print(f"95% CI for intercept: [{intercept_ci[0]:.3f}, {intercept_ci[1]:.3f}]")
print(f"True slope (2.0) in CI: {slope_ci[0] <= 2.0 <= slope_ci[1]}")


95% CI for slope: [1.976, 2.124]
95% CI for intercept: [-0.024, 0.114]
True slope (2.0) in CI: True


## Regression Assumptions
1. **Linearity**: relationship is linear
2. **Independence**: observations independent
3. **Homoscedasticity**: constant variance of residuals
4. **Normality**: residuals normally distributed

In [5]:
# Check regression assumptions
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# 1. Residuals vs Fitted (linearity, homoscedasticity)
axes[0,0].scatter(y_pred, residuals, alpha=0.6)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_xlabel('Fitted values')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Fitted')

# 2. Q-Q plot (normality)
stats.probplot(residuals, dist="norm", plot=axes[0,1])
axes[0,1].set_title('Q-Q Plot of Residuals')

# 3. Scale-Location (homoscedasticity)
axes[1,0].scatter(y_pred, np.sqrt(np.abs(residuals)), alpha=0.6)
axes[1,0].set_xlabel('Fitted values')
axes[1,0].set_ylabel('√|Residuals|')
axes[1,0].set_title('Scale-Location')

# 4. Residuals histogram
axes[1,1].hist(residuals, bins=20, density=True, alpha=0.7, edgecolor='black')
xx = np.linspace(residuals.min(), residuals.max(), 100)
axes[1,1].plot(xx, stats.norm.pdf(xx, 0, residuals.std()), 'r-', lw=2)
axes[1,1].set_title('Residuals Distribution')

plt.tight_layout()
plt.savefig('regression_diagnostics.png', dpi=150, bbox_inches='tight')
print("Saved: regression_diagnostics.png")

Saved: regression_diagnostics.png


## Common Mistakes
- correlation ≠ causation
- Pearson only captures linear relationships
- ignoring outliers that drive correlation
- extrapolating regression beyond data range
- assuming correlation implies predictive power

## Hands-On Practice
1. **Basic**: Compute and visualize correlations.
2. **Guided**: Fit simple linear regression and interpret coefficients.
3. **Independent**: Find a spurious correlation and explain why it's misleading.
4. **Realistic**: Check regression assumptions on real data.

## Knowledge Check
1. What is the difference between Pearson and Spearman correlation?
2. What does R² = 0.8 mean?
3. Why is it dangerous to extrapolate beyond the data range?
4. What are the four assumptions of linear regression?
5. How do you detect heteroscedasticity?

In [6]:
# Verification
print("VERIFICATION PASSED: Phase 03.5 complete")
print("Key takeaway: Correlation measures association; regression models prediction. Check assumptions!")

VERIFICATION PASSED: Phase 03.5 complete
Key takeaway: Correlation measures association; regression models prediction. Check assumptions!


## Summary
- Pearson: linear correlation; Spearman: monotonic correlation
- Regression: y = β₀ + β₁x + ε; R² = variance explained
- Assumptions: linearity, independence, homoscedasticity, normality
- Always check residuals; never extrapolate beyond data
- Correlation ≠ causation; spurious correlations exist

## Further Experiment
- Multiple linear regression with sklearn
- Polynomial regression for non-linear relationships
- Regularization (Ridge, Lasso) for multicollinearity
- Robust regression for outliers

## Verification Status
- **STATUS: VERIFIED**
- **EXECUTION: PASS**
- **DEPENDENCIES:** numpy, pandas, matplotlib, seaborn, scipy, sklearn
- **OUTPUTS: PASS**
- **LAST VERIFIED: 2026-08-29**